In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import opendatasets as od
import os

def main():
    # 1. FETCH DATASET FROM KAGGLE
    print("Connecting to Kaggle to download the dataset...")
    dataset_url = "https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces"
    od.download(dataset_url)

    # Define where the downloaded images are stored
    # The 'real_vs_fake' folder contains 'real' and 'fake' subfolders
    base_dir = './140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train'

    # 2. DATA PREPARATION (IMAGE GENERATORS)
    print("\nSetting up image data generators...")

    # Scale pixels to be between 0 and 1, and split 20% for validation
    datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

    # Load training images
    train_generator = datagen.flow_from_directory(
        base_dir,
        target_size=(128, 128), # Resize all images to 128x128
        batch_size=32,
        class_mode='binary',    # 0 for fake, 1 for real
        subset='training'
    )

    # Load validation images
    validation_generator = datagen.flow_from_directory(
        base_dir,
        target_size=(128, 128),
        batch_size=32,
        class_mode='binary',
        subset='validation'
    )

    # 3. BUILD THE NEURAL NETWORK (CNN)
    print("\nBuilding the Deepfake Detection CNN...")
    model = Sequential([
        # Layer 1: Detect basic edges
        Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
        MaxPooling2D(2, 2),

        # Layer 2: Detect complex textures (blurring, weird lighting)
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),

        # Layer 3: Deep feature extraction
        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),

        # Flatten the 2D images into a 1D array for the final decision
        Flatten(),

        # Dense network
        Dense(128, activation='relu'),
        Dropout(0.5), # Drops 50% of neurons randomly to prevent overfitting

        # Final Output Layer (Sigmoid gives a probability between 0 and 1)
        Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    # 4. TRAINING THE AI
    print("\nStarting training... (This will take time depending on your hardware)")

    history = model.fit(
        train_generator,
        epochs=5, # Number of times to loop through the dataset. Keep low for testing.
        validation_data=validation_generator
    )

    # 5. SAVING THE MODEL
    print("\nTraining complete! Saving the model...")
    model.save('deepfake_detector.h5')
    print("Success! Model saved as 'deepfake_detector.h5'. You can now use it for predictions.")

if __name__ == "__main__":
    main()

Connecting to Kaggle to download the dataset...
Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: deepaksehrawat99
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces


100%|██████████| 3.75G/3.75G [00:35<00:00, 112MB/s]




Setting up image data generators...
Found 80000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.

Building the Deepfake Detection CNN...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Starting training... (This will take time depending on your hardware)
Epoch 1/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 174s 67ms/step - accuracy: 0.7137 - loss: 0.5501 - val_accuracy: 0.8210 - val_loss: 0.4010
Epoch 2/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 148s 59ms/step - accuracy: 0.8478 - loss: 0.3515 - val_accuracy: 0.8910 - val_loss: 0.2709
Epoch 3/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 123s 49ms/step - accuracy: 0.9014 - loss: 0.2410 - val_accuracy: 0.9184 - val_loss: 0.2067
Epoch 4/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 126s 50ms/step - accuracy: 0.9277 - loss: 0.1832 - val_accuracy: 0.9166 - val_loss: 0.2070
Epoch 5/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 125s 50ms/step - accuracy: 0.9415 - loss: 0.1487 - val_accuracy: 0.9388 - val_loss: 0.1706



Training complete! Saving the model...
Success! Model saved as 'deepfake_detector.h5'. You can now use it for predictions.


In [2]:
pip install opendatasets

In [21]:
!pip install tf2onnx opendatasets -q

In [23]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import opendatasets as od
import os

def main():
    # 1. FETCH DATASET FROM KAGGLE
    print("Connecting to Kaggle to download the dataset...")
    dataset_url = "https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces"
    od.download(dataset_url)

    base_dir = './140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train'

    # 2. DATA PREPARATION
    print("\nSetting up image data generators...")
    datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

    train_generator = datagen.flow_from_directory(
        base_dir,
        target_size=(128, 128),
        batch_size=32,
        class_mode='binary',
        subset='training'
    )

    validation_generator = datagen.flow_from_directory(
        base_dir,
        target_size=(128, 128),
        batch_size=32,
        class_mode='binary',
        subset='validation'
    )

    # 3. BUILD THE CNN
    print("\nBuilding the Deepfake Detection CNN...")
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
        MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    # 4. TRAIN
    print("\nStarting training...")
    model.fit(
        train_generator,
        epochs=5,
        validation_data=validation_generator
    )

if __name__ == "__main__":
    main()

Connecting to Kaggle to download the dataset...
Skipping, found downloaded files in "./140k-real-and-fake-faces" (use force=True to force download)

Setting up image data generators...
Found 80000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.

Building the Deepfake Detection CNN...

Starting training...
Epoch 1/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 127s 49ms/step - accuracy: 0.7092 - loss: 0.5608 - val_accuracy: 0.8034 - val_loss: 0.4413
Epoch 2/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 119s 48ms/step - accuracy: 0.8237 - loss: 0.4017 - val_accuracy: 0.8673 - val_loss: 0.3140
Epoch 3/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 119s 48ms/step - accuracy: 0.8782 - loss: 0.2959 - val_accuracy: 0.9041 - val_loss: 0.2320
Epoch 4/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 121s 48ms/step - accuracy: 0.9060 - loss: 0.2331 - val_accuracy: 0.9044 - val_loss: 0.2370
Epoch 5/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 123s 49ms/step - accuracy: 0.9235 - loss: 0.1922 - val_accuracy: 0.9179 - val_loss: 0.2124


In [29]:
print("\nSaving model...")
model.save('/content/deepfake_detector.keras')
model.save('/content/deepfake_detextor.h5')
model.export('/content/deepfake_detector')
print("Saved!")


Saving model...
Saved artifact at '/content/deepfake_detector.onnx'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  135948356073040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135948356073424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135948356072464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135948356069968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135948356070160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135948356074000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135948356074576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135948356074384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135948356075152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135948356075344: TensorSpec(shape=(), dtype=tf.resour

256

In [30]:
!python -m tf2onnx.convert \
  --saved-model /content/deepfake_detector/ \
  --output /content/deepfake_detector.onnx \
  --opset 13

I0000 00:00:1780758847.468936   16682 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 12097 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
2026-06-06 15:14:07,470 - WARNING - '--tag' not specified for saved_model. Using --tag serve
2026-06-06 15:14:07,597 - INFO - Signatures found in model: [serve,serving_default].
2026-06-06 15:14:07,597 - WARNING - '--signature_def' not specified, using first signature: serve
2026-06-06 15:14:07,597 - INFO - Output names: ['output_0']
I0000 00:00:1780758847.600530   16682 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1780758847.600747   16682 single_machine.cc:376] Starting new session
I0000 00:00:1780758847.607667   16682 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 12097 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:178